# 🇻🇳 Hệ thống Nhận diện Thực thể có Tên (Vietnamese NER)

**Mô hình**: BiLSTM kết hợp từ điển đặc trưng (Gazetteer).

## 1. Import Thư viện & Cài đặt Thiết bị (Device)
Khởi tạo các thư viện cần thiết và kiểm tra GPU.

In [13]:
import ast
import json
import random
import logging
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

from underthesea import word_tokenize
from TorchCRF import CRF
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from seqeval.scheme import IOB2

# Thiết lập Log
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger()

# Cố định Seed để kết quả có thể tái lập
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Thiết lập GPU (nếu có)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Đang sử dụng thiết bị: {device}")

🚀 Đang sử dụng thiết bị: cpu


## 2. Cấu hình (Config) & Từ điển (Gazetteer)
Định nghĩa các nhãn (tags), siêu tham số (hyperparameters), và danh sách từ khóa hỗ trợ mô hình.

In [11]:
# =========================
# BỘ NHÃN (TAGS)
# =========================
LABEL_MAP: Dict[int, str] = {
    0: "O",
    1: "B-PER", 2: "I-PER",  # Person
    3: "B-ORG", 4: "I-ORG",  # Organization
    5: "B-LOC", 6: "I-LOC",  # Location
    7: "B-MISC", 8: "I-MISC", # Miscellaneous
}
TAG_LIST: List[str] = [LABEL_MAP[i] for i in sorted(LABEL_MAP)]
TAG_TO_ID: Dict[str, int] = {tag: idx for idx, tag in enumerate(TAG_LIST)}
ID_TO_TAG: Dict[int, str] = {idx: tag for tag, idx in TAG_TO_ID.items()}

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_LABEL = TAG_TO_ID["O"]

# =========================
# TỪ ĐIỂN (GAZETTEER)
# =========================
PERSON_PREFIXES: set[str] = {
    "ông", "bà", "anh", "chị", "em", "cô", "chú", "bác", "giáo_sư", "tiến_sĩ",
    "thạc_sĩ", "kỹ_sư", "tổng_giám_đốc", "giám_đốc", "chủ_tịch", "phó_chủ_tịch",
    "bộ_trưởng", "thứ_trưởng", "tổng_bí_thư", "thủ_tướng", "chủ_tịch_nước", "đại_tướng"
}
LOCATION_GAZETTEER: set[str] = {
    "hà_nội", "hồ_chí_minh", "sài_gòn", "hải_phòng", "đà_nẵng", "cần_thơ",
    "nghệ_an", "thanh_hóa", "huế", "nha_trang", "đà_lạt", "vũng_tàu", "việt_nam",
    "tây_nguyên", "đồng_bằng_sông_cửu_long", "trường_sa", "hoàng_sa", "hạ_long"
}
ORG_GAZETTEER: set[str] = {
    "vng", "viettel", "fpt", "vinamilk", "vingroup", "masan", "mobifone",
    "vinaphone", "vietcombank", "bidv", "vietinbank", "agribank",
    "bộ_y_tế", "bộ_giáo_dục", "chính_phủ", "quốc_hội", "nhà_nước", "đảng"
}

# =========================
# SIÊU THAM SỐ (DATACLASS)
# =========================
@dataclass
class ModelConfig:
    vocab_size: int
    tag_size: int = len(TAG_LIST)
    embedding_dim: int = 200
    hidden_dim: int = 256
    num_layers: int = 2
    dropout: float = 0.3
    pad_idx: int = 0
    use_highway: bool = False
    embedding_norm: bool = False
    variational_dropout: float = 0.0
    use_gazetteer: bool = True  # Bật tính năng nối Gazetteer
    gazetteer_dim: int = 3      # Số chiều (Person, Location, Org)

## 3. Tiền xử lý dữ liệu (Dataset & DataLoader)
Hàm trích xuất Gazetteer, đọc file CSV và sinh các Tensor.

In [9]:
def tokenize_text(text: str, lowercase: bool = False) -> List[str]:
    """Tách từ bằng underthesea."""
    output = word_tokenize(text, format="text")
    tokens = [token for token in output.split() if token.strip()]
    return [t.lower() if lowercase else t for t in tokens]

def build_gazetteer_feats(tokens: List[str]) -> List[List[float]]:
    """Sinh đặc trưng Gazetteer (3 chiều) cho từng token trong câu."""
    feats = []
    for t in tokens:
        tl = t.lower()
        feats.append([
            float(tl in PERSON_PREFIXES),
            float(tl in LOCATION_GAZETTEER),
            float(tl in ORG_GAZETTEER),
        ])
    return feats

def parse_json_list(value: object) -> List[object]:
    if isinstance(value, list): return value
    if isinstance(value, str): return ast.literal_eval(value)
    raise ValueError(f"Invalid JSON: {value}")

def load_ner_csv(path: Path, lowercase: bool = False):
    """Đọc file CSV và trả về (tokens, tag_ids)"""
    df = pd.read_csv(path)
    examples = []
    for _, row in df.iterrows():
        try:
            raw_tokens = parse_json_list(row["tokens"])
            raw_tags = parse_json_list(row["ner_tags"])
        except: continue
        
        if len(raw_tokens) != len(raw_tags): continue
        
        tokens, tag_ids, is_valid = [], [], True
        for token, tag in zip(raw_tokens, raw_tags):
            tokens.append(str(token).lower() if lowercase else str(token))
            
            # Chuẩn hóa tag thành ID
            if isinstance(tag, int): tag_id = tag
            elif isinstance(tag, str) and tag.isdigit(): tag_id = int(tag)
            elif isinstance(tag, str) and tag in TAG_TO_ID: tag_id = TAG_TO_ID[tag]
            else: is_valid = False; break
            
            tag_ids.append(tag_id)
            
        if is_valid: examples.append((tokens, tag_ids))
    return examples

def build_token_vocab(sentences, min_freq=1):
    """Xây dựng từ điển token (Vocabulary)."""
    counter = Counter(token for tokens, _ in sentences for token in tokens)
    tokens = [PAD_TOKEN, UNK_TOKEN] + [token for token, count in counter.most_common() if count >= min_freq]
    token2id = {token: idx for idx, token in enumerate(tokens)}
    return token2id, {idx: token for token, idx in token2id.items()}

class NERDataset(Dataset):
    def __init__(self, examples, token2id):
        self.records = []
        unk_idx = token2id[UNK_TOKEN]
        for tokens, tags in examples:
            token_ids = [token2id.get(t, unk_idx) for t in tokens]
            gaz_feats = build_gazetteer_feats(tokens)
            self.records.append({"tokens": token_ids, "tags": tags, "raw_tokens": tokens, "gaz_feats": gaz_feats})

    def __len__(self): return len(self.records)
    def __getitem__(self, i): return self.records[i]

def collate_fn(batch):
    """Pad các câu trong batch cho bằng nhau."""
    batch = sorted(batch, key=lambda r: len(r["tokens"]), reverse=True)
    token_seqs = [torch.tensor(r["tokens"], dtype=torch.long) for r in batch]
    tag_seqs = [torch.tensor(r["tags"], dtype=torch.long) for r in batch]
    gaz_seqs = [torch.tensor(r["gaz_feats"], dtype=torch.float) for r in batch]
    raw_tokens = [r["raw_tokens"] for r in batch]
    
    lengths = torch.tensor([seq.size(0) for seq in token_seqs], dtype=torch.long)
    tokens_padded = pad_sequence(token_seqs, batch_first=True, padding_value=0)
    tags_padded = pad_sequence(tag_seqs, batch_first=True, padding_value=PAD_LABEL)
    gaz_padded = pad_sequence(gaz_seqs, batch_first=True, padding_value=0.0)
    
    mask = torch.arange(tokens_padded.size(1), dtype=torch.long).unsqueeze(0) < lengths.unsqueeze(1)
    return tokens_padded, tags_padded, lengths, mask.bool(), raw_tokens, gaz_padded

# Đọc dữ liệu (Đảm bảo file merged_train.csv có sẵn trong folder data/)
try:
    train_examples = load_ner_csv(Path("data/merged_train.csv"))
    valid_examples = load_ner_csv(Path("data/merged_valid.csv"))
    print(f"Train size: {len(train_examples)} | Valid size: {len(valid_examples)}")
    token2id, id2token = build_token_vocab(train_examples, min_freq=1)
    train_loader = DataLoader(NERDataset(train_examples, token2id), batch_size=32, shuffle=True, collate_fn=collate_fn)
    valid_loader = DataLoader(NERDataset(valid_examples, token2id), batch_size=32, shuffle=False, collate_fn=collate_fn)
except Exception as e:
    print(f"⚠️ Không thể load dữ liệu: {e}. Vui lòng kiểm tra lại thư mục data/")
    token2id = {} # Fallback rỗng


Train size: 18513 | Valid size: 6372


## 4. Đo lường Hiệu suất (Metrics)
Tính F1-score ở cấp độ thực thể (Entity-level) bằng seqeval.

In [8]:
def ids_to_tags(sequences: Sequence[Sequence[int]]) -> List[List[str]]:
    return [[ID_TO_TAG[int(t_id)] for t_id in seq] for seq in sequences]

def compute_entity_metrics(true_ids, pred_ids):
    """Tính toán P/R/F1 theo scheme IOB2"""
    true_tags = ids_to_tags(true_ids)
    pred_tags = ids_to_tags(pred_ids)
    return {
        "entity_precision": precision_score(true_tags, pred_tags, zero_division=0),
        "entity_recall": recall_score(true_tags, pred_tags, zero_division=0),
        "entity_f1": f1_score(true_tags, pred_tags, zero_division=0),
    }

def get_classification_report(true_ids, pred_ids):
    true_tags = ids_to_tags(true_ids)
    pred_tags = ids_to_tags(pred_ids)
    return classification_report(true_tags, pred_tags, digits=4, scheme=IOB2)

## 5. Xây dựng Kiến trúc Mô hình (BiLSTM-CRF)
Bao gồm: Embedding $\rightarrow$ Concat Gazetteer $\rightarrow$ BiLSTM $\rightarrow$ Linear $\rightarrow$ CRF.

In [7]:
class Highway(nn.Module):
    def __init__(self, size: int, num_layers: int = 1):
        super().__init__()
        self.nonlinear = nn.ModuleList([nn.Linear(size, size) for _ in range(num_layers)])
        self.gate = nn.ModuleList([nn.Linear(size, size) for _ in range(num_layers)])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for nonlin, gate in zip(self.nonlinear, self.gate):
            n = torch.relu(nonlin(x))
            g = torch.sigmoid(gate(x))
            x = g * n + (1 - g) * x
        return x

class BiLSTMCRF(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.embedding_dim, padding_idx=cfg.pad_idx)
        self.highway = Highway(cfg.embedding_dim) if cfg.use_highway else nn.Identity()
        
        # Tăng kích thước đầu vào LSTM nếu dùng Gazetteer
        lstm_input_size = cfg.embedding_dim + (cfg.gazetteer_dim if cfg.use_gazetteer else 0)
        
        self.lstm = nn.LSTM(
            input_size=lstm_input_size,
            hidden_size=cfg.hidden_dim,
            num_layers=cfg.num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=cfg.dropout if cfg.num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(cfg.dropout)
        self.classifier = nn.Linear(cfg.hidden_dim * 2, cfg.tag_size)
        self.crf = CRF(cfg.tag_size, batch_first=True)
        self._reset_parameters()

    def _reset_parameters(self):
        """Khởi tạo trọng số Xavier cho các layer."""
        nn.init.xavier_uniform_(self.embedding.weight)
        for name, param in self.lstm.named_parameters():
            if "weight" in name: nn.init.xavier_uniform_(param)
            elif "bias" in name: nn.init.zeros_(param)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def _compute_emissions(self, token_ids, lengths, gaz_feats=None):
        """Tính toán điểm logits cho mỗi token trước khi truyền qua CRF."""
        embedded = self.embedding(token_ids)
        embedded = self.highway(embedded)
        
        # Nối đặc trưng Gazetteer vào Word Embeddings
        if gaz_feats is not None and getattr(self.cfg, "use_gazetteer", False):
            embedded = torch.cat([embedded, gaz_feats], dim=-1)
            
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(packed_output, batch_first=True)
        output = self.dropout(output)
        return self.classifier(output)

    def forward(self, token_ids, lengths, tags, mask, gaz_feats=None):
        """Tính Loss: Negative Log-Likelihood qua hàm CRF."""
        emissions = self._compute_emissions(token_ids, lengths, gaz_feats)
        return -self.crf(emissions, tags, mask=mask, reduction="mean")

    def decode(self, token_ids, lengths, mask, gaz_feats=None):
        """Dự đoán nhãn bằng thuật toán Viterbi (có sẵn trong CRF)."""
        emissions = self._compute_emissions(token_ids, lengths, gaz_feats)
        return self.crf.decode(emissions, mask=mask)

if token2id:
    model_cfg = ModelConfig(vocab_size=len(token2id))
    model = BiLSTMCRF(model_cfg).to(device)
    print("✅ Mô hình đã được khởi tạo trên", device)

✅ Mô hình đã được khởi tạo trên cpu


## 6. Huấn luyện Mô hình (Training Loop)
Huấn luyện với bộ tối ưu `AdamW`, lập lịch `OneCycleLR`, và `Autocast` (Mixed Precision).

In [6]:
def train_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    for tokens, tags, lengths, mask, raw, gaz in loader:
        tokens, tags, lengths, mask, gaz = tokens.to(device), tags.to(device), lengths.to(device), mask.to(device), gaz.to(device)
        
        with torch.amp.autocast('cuda', enabled=scaler is not None):
            loss = model(tokens, lengths, tags, mask, gaz)
            
        if scaler:
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
        if scheduler: scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item() * tokens.size(0)
        
    return total_loss / len(loader.dataset)

def eval_model(model, loader):
    model.eval()
    total_loss, all_true, all_pred = 0, [], []
    
    with torch.no_grad():
        for tokens, tags, lengths, mask, raw, gaz in loader:
            tokens, tags, lengths, mask, gaz = tokens.to(device), tags.to(device), lengths.to(device), mask.to(device), gaz.to(device)
            
            loss = model(tokens, lengths, tags, mask, gaz)
            total_loss += loss.item() * tokens.size(0)
            
            preds = model.decode(tokens, lengths, mask, gaz)
            for p, g, l in zip(preds, tags.cpu().tolist(), lengths.cpu().tolist()):
                all_pred.append(p[:l])
                all_true.append(g[:l])
                
    metrics = compute_entity_metrics(all_true, all_pred)
    return total_loss / len(loader.dataset), metrics["entity_f1"], get_classification_report(all_true, all_pred)

In [7]:
if token2id:
    # Khởi tạo tham số huấn luyện
    epochs = 20
    lr = 3e-4
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = OneCycleLR(optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=len(train_loader), pct_start=0.1)
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
    
    best_f1, best_epoch, patience_counter = 0.0, 0, 0
    history = {"train": [], "val": [], "f1": []}
    Path("models").mkdir(exist_ok=True)

    #  Train trực tiếp
    
    print("🔥 Bắt đầu Training...")
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler)
        val_loss, val_f1, report = eval_model(model, valid_loader)
        
        history["train"].append(train_loss)
        history["val"].append(val_loss)
        history["f1"].append(val_f1)
        print(f"Epoch {epoch}: Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f} | F1={val_f1:.4f}")
        
        if val_f1 > best_f1:
            best_f1, best_epoch, patience_counter = val_f1, epoch, 0
            
            # Lưu checkpoint
            torch.save({
                "model_state_dict": model.state_dict(), 
                "token2id": token2id, 
                "config": asdict(model.cfg)
            }, "models/best_model_notebook.pt")
            print(f"  ⭐ Đã lưu best model mới!")
        else:
            patience_counter += 1
            if patience_counter >= 3:
                print("🛑 Dừng sớm (Early Stopping)")
                break
    

🔥 Bắt đầu Training...


/opt/anaconda3/envs/vietnlp/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 1: Train Loss=23.8568 | Val Loss=12.8380 | F1=0.1970
  ⭐ Đã lưu best model mới!
Epoch 2: Train Loss=4.9415 | Val Loss=6.0833 | F1=0.6451
  ⭐ Đã lưu best model mới!
Epoch 3: Train Loss=2.1663 | Val Loss=3.9979 | F1=0.7862
  ⭐ Đã lưu best model mới!
Epoch 4: Train Loss=1.2197 | Val Loss=3.4955 | F1=0.8173
  ⭐ Đã lưu best model mới!
Epoch 5: Train Loss=0.8597 | Val Loss=3.6424 | F1=0.8291
  ⭐ Đã lưu best model mới!
Epoch 6: Train Loss=0.6306 | Val Loss=3.8885 | F1=0.8465
  ⭐ Đã lưu best model mới!
Epoch 7: Train Loss=0.5074 | Val Loss=3.9099 | F1=0.8527
  ⭐ Đã lưu best model mới!
Epoch 8: Train Loss=0.3928 | Val Loss=3.8775 | F1=0.8554
  ⭐ Đã lưu best model mới!
Epoch 9: Train Loss=0.3063 | Val Loss=4.1216 | F1=0.8566
  ⭐ Đã lưu best model mới!
Epoch 10: Train Loss=0.2596 | Val Loss=4.5099 | F1=0.8586
  ⭐ Đã lưu best model mới!
Epoch 11: Train Loss=0.2124 | Val Loss=4.7239 | F1=0.8558
Epoch 12: Train Loss=0.1697 | Val Loss=4.7045 | F1=0.8560
Epoch 13: Train Loss=0.1384 | Val Loss=5.

## 7. Suy luận (Inference)
Trích xuất thực thể trực tiếp từ một đoạn văn bản nhập vào.

In [25]:
from pathlib import Path
import torch


# =========================================================
# NER DEBUG INFERENCE
# =========================================================

def predict_ner_debug(
    text: str,
    model_path: str = "models/best_model_notebook.pt"
):
    """
    Debug toàn bộ pipeline NER:
    - Tokens
    - Token IDs
    - Gazetteer features
    - BIO predictions
    - Final entities
    - Statistics
    """

    # =====================================================
    # CHECK MODEL
    # =====================================================

    if not Path(model_path).exists():

        print("❌ Không tìm thấy model checkpoint")
        print(f"📁 Path: {model_path}")

        return

    # =====================================================
    # LOAD MODEL
    # =====================================================

    print("\n🚀 Loading model...")

    ckpt = torch.load(
        model_path,
        map_location=device,
        weights_only=True
    )

    t2id = ckpt["token2id"]

    cfg_dict = ckpt["config"]

    # Backward compatibility
    if "use_gazetteer" not in cfg_dict:

        cfg_dict["use_gazetteer"] = False
        cfg_dict["gazetteer_dim"] = 0

    # =====================================================
    # INIT MODEL
    # =====================================================

    infer_model = BiLSTMCRF(
        ModelConfig(**cfg_dict)
    ).to(device)

    infer_model.load_state_dict(
        ckpt["model_state_dict"]
    )

    infer_model.eval()

    print("✅ Model loaded successfully")

    # =====================================================
    # TOKENIZE
    # =====================================================

    tokens = tokenize_text(text)

    # =====================================================
    # TOKEN IDS
    # =====================================================

    token_ids = [

        t2id.get(
            token,
            t2id.get(UNK_TOKEN, 1)
        )

        for token in tokens
    ]

    # =====================================================
    # GAZETTEER FEATURES
    # =====================================================

    gaz_feats = build_gazetteer_feats(tokens)

    # =====================================================
    # PRINT TOKEN TABLE
    # =====================================================

    print("\n" + "=" * 120)
    print("🔍 TOKEN DEBUG TABLE")
    print("=" * 120)

    print(
        f"{'IDX':<5}"
        f"{'TOKEN':<35}"
        f"{'TOKEN_ID':<12}"
        f"{'GAZETTEER'}"
    )

    print("-" * 120)

    for idx, (tok, tid, gaz) in enumerate(
        zip(tokens, token_ids, gaz_feats)
    ):

        print(
            f"{idx:<5}"
            f"{tok:<35}"
            f"{tid:<12}"
            f"{gaz}"
        )

    # =====================================================
    # BUILD TENSORS
    # =====================================================

    t_ids = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=device
    )

    t_len = torch.tensor(
        [len(token_ids)],
        dtype=torch.long,
        device=device
    )

    t_gaz = torch.tensor(
        [gaz_feats],
        dtype=torch.float,
        device=device
    )

    mask = torch.ones(
        (1, len(token_ids)),
        dtype=torch.bool,
        device=device
    )

    # =====================================================
    # INFERENCE
    # =====================================================

    print("\n🧠 Running inference...")

    with torch.no_grad():

        pred_ids = infer_model.decode(
            t_ids,
            t_len,
            mask,
            t_gaz
        )[0]

    pred_tags = [

        ID_TO_TAG.get(tag_id, "O")

        for tag_id in pred_ids
    ]

    # =====================================================
    # PRINT BIO OUTPUT
    # =====================================================

    print("\n" + "=" * 120)
    print("🧬 BIO TAG OUTPUT")
    print("=" * 120)

    print(
        f"{'IDX':<5}"
        f"{'TOKEN':<35}"
        f"{'BIO TAG'}"
    )

    print("-" * 120)

    for idx, (tok, tag) in enumerate(
        zip(tokens, pred_tags)
    ):

        print(
            f"{idx:<5}"
            f"{tok:<35}"
            f"{tag}"
        )

    # =====================================================
    # BUILD ENTITIES
    # =====================================================

    entities = []

    current_entity = None

    for token, tag in zip(tokens, pred_tags):

        # -------------------------------------------------
        # OUTSIDE
        # -------------------------------------------------

        if tag == "O":

            if current_entity:

                entities.append(current_entity)

                current_entity = None

            continue

        # -------------------------------------------------
        # INVALID TAG
        # -------------------------------------------------

        if "-" not in tag:

            continue

        prefix, label = tag.split("-", 1)

        # -------------------------------------------------
        # BEGIN ENTITY
        # -------------------------------------------------

        if (

            prefix == "B"

            or current_entity is None

            or current_entity["label"] != label
        ):

            if current_entity:

                entities.append(current_entity)

            current_entity = {

                "text": token,

                "label": label
            }

        # -------------------------------------------------
        # INSIDE ENTITY
        # -------------------------------------------------

        else:

            current_entity["text"] += " " + token

    # Append last entity
    if current_entity:

        entities.append(current_entity)

    # =====================================================
    # PRINT ENTITIES
    # =====================================================

    print("\n" + "=" * 120)
    print("🏷️ DETECTED ENTITIES")
    print("=" * 120)

    if len(entities) == 0:

        print("❌ No entities detected")

    else:

        for idx, ent in enumerate(entities, 1):

            print(
                f"[{idx:02d}] "
                f"{ent['text']:<60}"
                f" -> [{ent['label']}]"
            )

    # =====================================================
    # ENTITY STATISTICS
    # =====================================================

    stats = {}

    for ent in entities:

        label = ent["label"]

        stats[label] = stats.get(label, 0) + 1

    print("\n" + "=" * 120)
    print("📊 ENTITY STATISTICS")
    print("=" * 120)

    total_entities = 0

    for label, count in stats.items():

        print(f"{label:<10}: {count}")

        total_entities += count

    print("-" * 120)

    print(f"TOTAL ENTITIES: {total_entities}")

    # =====================================================
    # RETURN
    # =====================================================

    return {

        "tokens": tokens,

        "token_ids": token_ids,

        "gazetteer_features": gaz_feats,

        "bio_tags": pred_tags,

        "entities": entities
    }


# =========================================================
# TEST
# =========================================================

text_test = """Hôm qua, Bộ Giáo dục Việt Nam đã tổ chức hội nghị tại Hà Nội để bàn về chương trình chuyển đổi số trong trường học.

Giám đốc Nguyễn Văn Nam cho biết thành phố Đà Nẵng sẽ phối hợp với Đại học Bách Khoa Hà Nội để nghiên cứu công nghệ trí tuệ nhân tạo.

Trong chuyến công tác tại Thành phố Hồ Chí Minh, Thủ tướng Phạm Minh Chính đã gặp Chủ tịch Nguyễn Thị Lan để trao đổi về dự án đường sắt cao tốc Hà Nội - Hải Phòng.

Ủy ban Nhân dân tỉnh Quảng Ninh cũng cam kết hỗ trợ doanh nghiệp địa phương phát triển du lịch thông minh tại Vịnh Hạ Long.

Trong khi đó, Tập đoàn Công nghệ Sao Việt đang mở rộng hợp tác với Công ty Phần mềm Minh Quân tại thị trường Cần Thơ và Đồng Nai.

Tổng Bí thư Nguyễn Phú Trọng cho biết Việt Nam sẽ tiếp tục triển khai vệ tinh viễn thông phục vụ khu vực Tây Nguyên, bao gồm cả Gia Lai và Đắk Lắk.
"""

# =========================================================
# RUN
# =========================================================

result = predict_ner_debug(text_test)


🚀 Loading model...
✅ Model loaded successfully

🔍 TOKEN DEBUG TABLE
IDX  TOKEN                              TOKEN_ID    GAZETTEER
------------------------------------------------------------------------------------------------------------------------
0    Hôm_qua                            4953        [0.0, 0.0, 0.0]
1    ,                                  2           [0.0, 0.0, 0.0]
2    Bộ                                 190         [0.0, 0.0, 0.0]
3    Giáo_dục                           3659        [0.0, 0.0, 0.0]
4    Việt_Nam                           2056        [0.0, 1.0, 0.0]
5    đã                                 14          [0.0, 0.0, 0.0]
6    tổ_chức                            607         [0.0, 0.0, 0.0]
7    hội_nghị                           2881        [0.0, 0.0, 0.0]
8    tại                                25          [0.0, 0.0, 0.0]
9    Hà_Nội                             400         [0.0, 1.0, 0.0]
10   để                                 43          [0.0, 0.0, 0.0]


In [30]:
# Mô hình nhận diện các thực thể trong đoạn văn bản test và in ra kết quả chi tiết.
# Các thực thể được in ra cùng với loại thực thể (Person, Location, Organization, Miscellaneous) và thống kê số lượng từng loại.
# Loại thực thể person đang bị miss khá nhiều, có thể do dữ liệu huấn luyện chưa đủ đa dạng hoặc model chưa học được tốt các đặc trưng của thực thể person. Cần xem xét lại dữ liệu và có thể thêm các kỹ thuật tăng cường dữ liệu hoặc điều chỉnh mô hình để cải thiện hiệu suất nhận diện thực thể person.
import pandas as pd
import ast

df = pd.read_csv("/Users/trananhviet/Documents/HUMG Uni/NLP/DIEM A/NLP/NLP/data/merged_train.csv")

for _, row in df.iterrows():

    tokens = ast.literal_eval(row["tokens"])
    tags = ast.literal_eval(row["ner_tags"])

    for t, tag in zip(tokens, tags):
        if t.lower() == "bao gồm":
            print(t, "->", tag)

In [36]:
# Kiểm tra tần suất token "bao gồm" trong tập dữ liệu huấn luyện và các nhãn đi kèm với nó để xem liệu mô hình có học được mối liên hệ giữa token này và các thực thể hay không.
import pandas as pd
import ast
from collections import Counter

# =========================
# CONFIG
# =========================
CSV_PATH = "data/merged_train.csv"

TARGET_TOKEN = "tại"

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(CSV_PATH)

next_token_labels = []
next_tokens = []

# =========================
# ANALYZE
# =========================
for _, row in df.iterrows():

    tokens = ast.literal_eval(row["tokens"])
    tags = ast.literal_eval(row["ner_tags"])

    for i in range(len(tokens) - 1):

        if tokens[i] == TARGET_TOKEN:

            next_tok = tokens[i + 1]
            next_tag = tags[i + 1]

            next_tokens.append(next_tok)
            next_token_labels.append(next_tag)

# =========================
# STATISTICS
# =========================
label_counter = Counter(next_token_labels)

total = sum(label_counter.values())

print("=" * 60)
print(f"ANALYSIS AFTER TOKEN: {TARGET_TOKEN}")
print("=" * 60)

for label, count in label_counter.items():

    ratio = count / total * 100

    print(f"{label:<10} : {count:<5} ({ratio:.2f}%)")

print("\n" + "=" * 60)
print("MOST COMMON NEXT TOKENS")
print("=" * 60)

token_counter = Counter(next_tokens)

for tok, count in token_counter.most_common(20):
    print(f"{tok:<30} {count}")

ANALYSIS AFTER TOKEN: tại
5          : 1367  (65.10%)
0          : 668   (31.81%)
3          : 63    (3.00%)
6          : 2     (0.10%)

MOST COMMON NEXT TOKENS
Bệnh                           483
nhà                            115
bệnh                           68
Đà                             67
khoa                           61
Trung                          60
các                            54
quận                           46
đây                            37
khu                            36
TP.                            30
VN                             30
Hà                             30
Trường                         29
Quảng                          28
TP                             28
Khoa                           28
sân                            27
Việt                           27
phường                         25


## Đánh giá mô hình với ví dụ trên

### Nhược điểm

- Dataset còn thiếu nhiều tên riêng trong tập từ vựng.  
  Ví dụ:
  
  - `Nguyễn_Phú_Trọng` bị gán `TOKEN_ID = 1` (`UNK`).
  - `Nguyễn_Văn_Nam` cũng bị gán `UNK`.

  Mặc dù token chức danh như:

  - `Tổng_Bí_thư`
  - `Giám_đốc`

  đã được thêm vào Gazetteer với nhãn `PER`, nhưng các token phía sau vẫn không được suy luận thành `I-PER`.

  Điều này cho thấy Gazetteer hiện tại chỉ hoạt động ở mức token-level, chưa hỗ trợ phrase-level hoặc context propagation.

---

- Xuất hiện hiện tượng dự đoán không nhất quán (inconsistency).

  Ví dụ:

  ```text
  Thủ_tướng -> O
  Phạm_Minh_Chính -> B-PER

- thực thể hành chính dài dễ bị tách sai token.
  
  ```text
  Ủy
  ban_Nhân_dân kiến cho Quảng_Ninh -> LOC
        

        